In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

import numpy as np

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "rnn"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
from pathlib import Path

greenhouse_path = Path("C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\teams_ghc.csv")
weather_path = Path("C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\weather_fill_missing_values.csv")
greenhouse_climate_path = Path("C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\greenhouse_climate_fill_missing_values.csv")
#root_zone_path = Path("C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\root_zone_fill_missing_values.csv")

greenhouse = pd.read_csv(greenhouse_path,parse_dates=["%time"]).set_index('%time')
weather=pd.read_csv(weather_path,parse_dates=["%time"]).set_index('%time')
greenhouse_climate=pd.read_csv(greenhouse_climate_path,parse_dates=["%time"]).set_index('%time')
#root_zone=pd.read_csv(root_zone_path,parse_dates=["%time"]).set_index('%time')

#pd.concatenate([greenhouse["days"],greenhouse["hours"],greenhouse["minutes"]],axis=1)

In [ ]:
greenhouse.head()

In [ ]:
greenhouse.columns

In [ ]:
#['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,          
#'Tot_PAR' , 'Tot_PAR_Lamps' , 'VentLee' , 'Ventwind' , 'assim_sp' , 'assim_vip' , 'co2_dos' , 'co2_sp' , 'co2_vip' ,      
#'dx_sp' , 'dx_vip' , 'int_blue_sp' , 'int_blue_vip' , 'int_farred_sp' , 'int_farred_vip' , 'int_red_sp' , 'int_red_vip',   
#'int_white_sp','int_white_vip','pH_drain_PC','scr_blck_sp',
#'scr_blck_vip','scr_enrg_sp','scr_enrg_vip','t_grow_min_sp','t_grow_min_vip','t_heat_sp','t_heat_vip',
#'t_rail_min_sp','t_rail_min_vip','t_vent_sp','t_ventlee_vip','t_ventwind_vip','water_sup_intervals_sp_min',
#'window_pos_lee_sp','window_pos_lee_vip','Water_Quantity','Duration_of_Irrigation','Irrigation_Time_Intervals']
setpoints=greenhouse[['co2_vip', 'dx_vip', 'int_blue_vip', 'int_farred_vip', 'int_red_vip',
       'int_white_vip', 'pH_drain_PC', 'scr_blck_vip', 'scr_enrg_vip',
       't_grow_min_vip', 't_heat_vip', 't_rail_min_vip', 't_ventlee_vip',
       't_ventwind_vip', 'water_sup', 'water_sup_intervals_vip_min',
       'window_pos_lee_vip', 'days']]
greenhouse.drop(['co2_vip', 'dx_vip', 'int_blue_vip', 'int_farred_vip', 'int_red_vip',
       'int_white_vip', 'pH_drain_PC', 'scr_blck_vip', 'scr_enrg_vip',
       't_grow_min_vip', 't_heat_vip', 't_rail_min_vip', 't_ventlee_vip',
       't_ventwind_vip', 'water_sup', 'water_sup_intervals_vip_min',
       'window_pos_lee_vip', 'days'],axis=1,inplace=True)

In [ ]:
setpoints.shape
setpoints.index

In [ ]:
greenhouse.shape


In [ ]:
weather.shape
teams_weather = pd.concat([weather] * 6, ignore_index=True).set_index(setpoints.index)

# Verify the shape of the new DataFrame
print(teams_weather.shape)

In [ ]:
teams_weather.index

In [ ]:
w_sp_data = pd.concat([teams_weather, setpoints], axis=1)

# Verify the shape of the merged DataFrame
print(w_sp_data.shape)

In [ ]:
w_sp_data.columns

In [ ]:
w_sp_data['days']

In [ ]:
##['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,          
##'Tot_PAR' , 'Tot_PAR_Lamps' , 'VentLee' , 'Ventwind' , 'assim_sp' , 'assim_vip' , 'co2_dos' , 'co2_sp' , 'co2_vip' ,      
##'dx_sp' , 'dx_vip' , 'int_blue_sp' , 'int_blue_vip' , 'int_farred_sp' , 'int_farred_vip' , 'int_red_sp' , 'int_red_vip',   
##'int_white_sp','int_white_vip','pH_drain_PC','scr_blck_sp',
##'scr_blck_vip','scr_enrg_sp','scr_enrg_vip','t_grow_min_sp','t_grow_min_vip','t_heat_sp','t_heat_vip',
##'t_rail_min_sp','t_rail_min_vip','t_vent_sp','t_ventlee_vip','t_ventwind_vip','water_sup_intervals_sp_min',
##'window_pos_lee_sp','window_pos_lee_vip','Water_Quantity','Duration_of_Irrigation','Irrigation_Time_Intervals']
#greenhouse_sp=greenhouse.drop(['EC_slab1', 'EC_slab2', 'WC_slab1',
#       'WC_slab2', 't_slab1', 't_slab2', 'AssimLight', 'BlackScr', 'CO2air',
#       'EC_drain_PC', 'EnScr', 'HumDef', 'PipeGrow', 'PipeLow', 'Rhair',
#       'Tair', 'Tot_PAR', 'Tot_PAR_Lamps', 'VentLee', 'Ventwind', 
#       'assim_vip', 'co2_dos', 'co2_vip',  'dx_vip',
#        'int_blue_vip', 'int_farred_vip',
#       'int_red_vip',  'int_white_vip',
#       'pH_drain_PC', 'scr_blck_vip',
#       'scr_enrg_vip',  't_grow_min_vip', 
#       't_heat_vip',  't_rail_min_vip', 
#       't_ventlee_vip', 't_ventwind_vip', 
#       'window_pos_lee_vip', 'Water_Quantity',
#       'Duration_of_Irrigation', 'Irrigation_Time_Intervals'],axis=1)

In [ ]:
#greenhouse_sp.columns

In [ ]:
import matplotlib.pyplot as plt
w_sp_data, greenhouse
import seaborn as sns
correlation_matrix = greenhouse.corr()
corr={}
outputs=greenhouse.columns
features=w_sp_data.columns

merged_data = pd.concat([w_sp_data,greenhouse],axis=1).set_index(greenhouse.index)
print(merged_data.shape)
    #print(merged_data.head())
correlation_matrix = merged_data.corr()
# Plotting the correlation matrix
plt.figure(figsize=(15, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix Between SP Features and Crop Parameters')
plt.show()

for output in outputs:
        corr[output]=merged_data[merged_data.columns].corr()[output].sort_values(ascending=False)[weather.columns].sort_values(ascending=False)
        #corr[output].drop(outputs,axis=0,inplace=True)
        corr[output].plot(kind='bar',title=f'{output} Correlation with Outsid Weather',figsize=(18, 6),fontsize=12)
        plt.show()
        save_fig(f' {output} Correlation with Outsid Weather')
        


In [ ]:
# teams_ghc_weekly = merged_data.resample('W').mean()
# for control_feature in features:
#         plt.figure(figsize=(15, 6))
#         sns.lineplot(data=teams_ghc_weekly[control_feature],legend=True)           
#         sns.lineplot(data=teams_ghc_weekly[outputs])

#         plt.title(f'Time Series of {control_feature} and greenhouse climate')
#         plt.xlabel('Date')
#         plt.legend(loc='right')
#         plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
w_sp_ss=      MinMaxScaler()
gh_ss=     MinMaxScaler()


In [ ]:
#greenhouse[weather.columns+"_next_day"] = weather[weather.columns].shift(-1)
#gh_train = greenhouse["2019-12-16 00:00:00" : "2020-03-26 00:00:00"]
#gh_valid = greenhouse["2020-03-26 00:00:00":"2020-04-26 00:00:00"]
#gh_test = greenhouse ['2020-04-26 00:00:00':]
#
#
#w_train = weather["2019-12-16 00:00:00" : "2020-03-26 00:00:00"]
#w_valid = weather["2020-03-26 00:00:00":"2020-04-26 00:00:00"]
#w_test = weather['2020-04-26 00:00:00':]
#
#greenhouse_climate[weather.columns+"_next_day"] = weather[weather.columns].shift(-1)
#ghc_train = greenhouse_climate["2019-12-16 00:00:00" : "2020-03-30 00:00:00"]
#ghc_valid = greenhouse_climate["2020-03-30 00:00:00":"2020-04-30 00:00:00"]
#ghc_test = greenhouse_climate['2020-04-30 00:00:00':]

from sklearn.model_selection import train_test_split

#greenhouse_sp[weather.columns+"_next_day"] = weather[weather.columns].shift(-1)
#ghsp_train = greenhouse_sp["2019-12-16 00:00:00" : "2020-03-30 00:00:00"]
#ghsp_valid = greenhouse_sp["2020-03-30 00:00:00":"2020-04-30 00:00:00"]
#ghsp_test = greenhouse_sp['2020-04-30 00:00:00':]
train_data = train_test_split(w_sp_data, greenhouse, test_size=0.2, random_state=42)
#print(train_data[0].shape)
(X_train, X_test, y_train, y_test) = train_data[0],train_data[1],train_data[2],train_data[3]
#Data Normalization
print('X_train',X_train.shape)
print('X_test',X_test.shape)
print('y_train',y_train.shape)
print('y_test',y_test.shape)


In [ ]:
greenhouse_X_train = pd.DataFrame(w_sp_ss.fit_transform(X_train),columns=X_train.columns)
greenhouse_X_test= pd.DataFrame(w_sp_ss.transform(X_test),columns=X_test.columns)
greenhouse_y_train =  pd.DataFrame(gh_ss.fit_transform(y_train),columns=y_train.columns)
greenhouse_y_test =  pd.DataFrame(gh_ss.transform(y_test),columns=y_test.columns)



#Data Normalization
print('greenhouse_X_train',greenhouse_X_train.shape)
print('greenhouse_X_test',greenhouse_X_test.shape)
print('greenhouse_y_train',greenhouse_y_train.shape)
print('greenhouse_y_test',greenhouse_y_test.shape)



In [ ]:
w_sp_ss=      MinMaxScaler()
gh_ss=     MinMaxScaler()
greenhouse_X = pd.DataFrame(w_sp_ss.fit_transform(w_sp_data),columns=w_sp_data.columns).set_index(greenhouse.index)
greenhouse_y =  pd.DataFrame(gh_ss.fit_transform(greenhouse),columns=greenhouse.columns).set_index(greenhouse.index)
merged_data = pd.concat([greenhouse_X,greenhouse_y],axis=1).set_index(greenhouse.index)

teams_ghc_weekly = merged_data.resample('W').mean()
for control_feature in features:
    for output in outputs:
        plt.figure(figsize=(15, 6))
        
        # Plot control feature with a label
        plt.plot(teams_ghc_weekly[control_feature], label=f'{control_feature}')
        
        # Plot output feature with a label
        plt.plot(teams_ghc_weekly[output], label=f'{output}')
        
        plt.title(f'Time Series of {control_feature} and greenhouse climate {output}')
        plt.xlabel('Date')
        
        # Show the legend
        plt.legend()
        
        plt.show()

In [ ]:
greenhouse_X_train .head()

In [ ]:
# Check for duplicates
duplicates = w_sp_data.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

In [ ]:
# Check for duplicates
duplicates = df.duplicated()
print(f"Number of duplicate rows: {duplicates.sum()}")

# View the duplicate rows (if needed)
duplicate_rows = df[df.duplicated()]
print("Duplicate rows:")
print(duplicate_rows)

# Remove duplicate rows
df_cleaned = df.drop_duplicates()

# Verify the new shape of the DataFrame
print(f"Shape after removing duplicates: {df_cleaned.shape}")

In [ ]:
greenhouse_y_test.min()

In [ ]:
tf.random.set_seed(42)

MLP_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=[greenhouse_X_train.shape[1]]),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation="relu"),
    tf.keras.layers.Dense(128,activation="relu"),

    tf.keras.layers.Dense(17, activation="linear")
])

MLP_model.input



In [ ]:
tf.random.set_seed(42)

avg_LSTM_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=[greenhouse_X_train.shape[1],greenhouse_X_train.shape[-1]]),
    tf.keras.layers.Conv1D(filters=32, kernel_size=4, strides=1, padding="same",
                           activation="relu"),
    tf.keras.layers.LSTM(64,return_sequences=True),
    tf.keras.layers.LSTM(64,return_sequences=False),
    tf.keras.layers.Dense(17, activation="linear")
])
avg_LSTM_model.input_shape


In [ ]:
from tensorflow.keras.optimizers import Adam

def fit_and_evaluate(model, train_input,train_output, valid_input,valid_output, learning_rate,ckpt, epochs=10,batch_size=32,patience=10):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=patience, restore_best_weights=True)
    model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    ckpt, monitor="val_mae", save_best_only=True)
    opt = Adam(learning_rate=learning_rate, decay=learning_rate / epochs)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_input,train_output, validation_split=0.2, epochs=epochs,batch_size=batch_size,
                        callbacks=[early_stopping_cb,model_ckpt])
    valid_loss, valid_mae = model.evaluate(valid_input,valid_output)
    
    #import joblib
    #joblib.dump(model,f'{ckpt}_model.pkl')

    # Save the model using TensorFlow's save method
    model.save(f'{ckpt}_model.h5')
    #loaded_model=joblib.load(f'{ckpt}_model.pkl')

    #new_data=testX
    #predictions=loaded_model.predict(new_data)
    #print(predictions,testY)
    # Plot training history
    plt.plot(history.history['mae'], label='train_mae')
    plt.plot(history.history['val_mae'], label='val_mae')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    return valid_mae 

In [ ]:
#tf.keras.backend.clear_session()

fit_and_evaluate(MLP_model, greenhouse_X_train,greenhouse_y_train, greenhouse_X_test,greenhouse_y_test, 
                learning_rate=0.001,ckpt="ghc_mlp_model",epochs=100,patience=10)


In [ ]:
seq_length = 12*24*7
tf.random.set_seed(42)  
ghsp_train_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouseSP_train.to_numpy(),  
    targets=greenhouseSP_train[['assim_sp', 'co2_sp', 'dx_sp',
       'int_blue_sp', 'int_farred_sp', 'int_red_sp', 'int_white_sp',
       'scr_blck_sp', 'scr_enrg_sp', 't_grow_min_sp', 't_heat_sp',
       't_rail_min_sp', 't_vent_sp', 'water_sup_intervals_sp_min',
       'window_pos_lee_sp']][seq_length:],  # forecast only the Water_Quantity series
    sequence_length=seq_length,
    batch_size=32,
    shuffle=[True],
    seed=42
)
ghsp_valid_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouseSP_valid.to_numpy(),
    targets=greenhouseSP_valid[['assim_sp', 'co2_sp', 'dx_sp',
       'int_blue_sp', 'int_farred_sp', 'int_red_sp', 'int_white_sp',
       'scr_blck_sp', 'scr_enrg_sp', 't_grow_min_sp', 't_heat_sp',
       't_rail_min_sp', 't_vent_sp', 'water_sup_intervals_sp_min',
       'window_pos_lee_sp']][seq_length:],
    sequence_length=seq_length,
    batch_size=32
)
ghsp_test_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouseSP_test.to_numpy(),
    targets=greenhouseSP_test[['assim_sp', 'co2_sp', 'dx_sp',
       'int_blue_sp', 'int_farred_sp', 'int_red_sp', 'int_white_sp',
       'scr_blck_sp', 'scr_enrg_sp', 't_grow_min_sp', 't_heat_sp',
       't_rail_min_sp', 't_vent_sp', 'water_sup_intervals_sp_min',
       'window_pos_lee_sp']][seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

In [ ]:
seq_length = 12*24*7
tf.random.set_seed(42)  
w_train_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    weather_train,  
    targets=weather_train[:][seq_length:],  
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
w_valid_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    weather_valid,
    targets=weather_valid[:][seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
w_test_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    weather_test,
    targets=weather_test[:][seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)

In [ ]:

seq_length = 12*24*7
tf.random.set_seed(42)  
ghc_train_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouse_climate_train.to_numpy(),  
    targets=greenhouse_climate_train[['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 
                                      'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,'Tot_PAR' , 'Tot_PAR_Lamps' , 
                                      'VentLee' , 'Ventwind' , 'co2_dos','Water_Quantity','Duration_of_Irrigation',
                                      'Irrigation_Time_Intervals']][seq_length:], 
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
ghc_valid_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouse_climate_valid.to_numpy(),
    targets=greenhouse_climate_valid[['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 
                                      'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,'Tot_PAR' , 'Tot_PAR_Lamps' , 
                                      'VentLee' , 'Ventwind' , 'co2_dos','Water_Quantity','Duration_of_Irrigation',
                                      'Irrigation_Time_Intervals']][seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
ghc_test_mulvar_ds = tf.keras.utils.timeseries_dataset_from_array(
    greenhouse_climate_test.to_numpy(),
    targets=greenhouse_climate_test[['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 
                                      'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,'Tot_PAR' , 'Tot_PAR_Lamps' , 
                                      'VentLee' , 'Ventwind' , 'co2_dos','Water_Quantity','Duration_of_Irrigation',
                                      'Irrigation_Time_Intervals']][seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)

In [ ]:
tf.random.set_seed(42)  
ghsp_mulvar_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=[None, greenhouse_sp.shape[1]],return_sequences=True),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(15)
])

tf.random.set_seed(42)  
w_mulvar_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=[None, weather.shape[1]],return_sequences=True),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(weather.shape[1])
])

tf.random.set_seed(42)  
ghc_mulvar_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=[None, greenhouse_climate.shape[1]],return_sequences=True),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(18)
])

In [ ]:
from tensorflow.keras.optimizers.legacy import Adam

def fit_and_evaluate(model, train_set, valid_set, learning_rate,ckpt, epochs=10):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=2, restore_best_weights=True)
    model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    ckpt, monitor="val_mae", save_best_only=True)
    opt = Adam(learning_rate=learning_rate, decay=learning_rate / epochs)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb,model_ckpt])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae 

In [ ]:
#compiles, fits, and evaluates the model
fit_and_evaluate(ghsp_mulvar_model, ghsp_train_mulvar_ds, ghsp_valid_mulvar_ds,
                 learning_rate=0.05,ckpt="ghsp_mulvar_model")

In [ ]:
ghsp_mulvar_model.save('C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\growthdetection\\Model\\NextSetPointsForecasting.hdf')

In [ ]:
#compiles, fits, and evaluates the model
fit_and_evaluate(w_mulvar_model, w_train_mulvar_ds, w_valid_mulvar_ds,
                 learning_rate=0.05,ckpt="w_mulvar_model")

In [ ]:
ghsp_mulvar_model.save('C:\\Users\\user\\Desktop\\iman\\AGHC\\AutonomousGreenHouseChallenge\\Code\\Training\\growthdetection\\Model\\OutsideWeatherForecastingModel.hdf')

In [ ]:
#compiles, fits, and evaluates the model
fit_and_evaluate(ghc_mulvar_model, ghc_train_mulvar_ds, ghc_valid_mulvar_ds,
                 learning_rate=0.05,ckpt="ghc_mulvar_model")

In [ ]:
ghsp_valid_prediction = ghsp_mulvar_model.predict(ghsp_valid_mulvar_ds)
#gh_Y_preds_valid=ghc_standard_scaler.inverse_transform(gh_valid_prediction)

#(gh_valid_prediction)
for idx, name in enumerate(['assim_sp', 'co2_sp', 'dx_sp',
       'int_blue_sp', 'int_farred_sp', 'int_red_sp', 'int_white_sp',
       'scr_blck_sp', 'scr_enrg_sp', 't_grow_min_sp', 't_heat_sp',
       't_rail_min_sp', 't_vent_sp', 'water_sup_intervals_sp_min',
       'window_pos_lee_sp']):
    mae = tf.keras.metrics.mean_absolute_error(
        greenhouseSP_valid[name][seq_length:], ghsp_valid_prediction[:, idx])
    print(name, mae)

In [ ]:
w_valid_prediction = w_mulvar_model.predict(w_valid_mulvar_ds)
#w_Y_preds_valid=w_sp_ss.inverse_transform(w_valid_prediction)
for idx, name in enumerate(weather.columns):
    mae =  tf.keras.metrics.mean_absolute_error(
        weather_valid[name][seq_length:], w_valid_prediction[:, idx])
    print(name, mae)

In [ ]:
ghc_valid_prediction = ghc_mulvar_model.predict(ghc_valid_mulvar_ds)
#ghc_Y_preds_valid=ghc_standard_scaler.inverse_transform(ghc_valid_prediction)
for idx, name in enumerate(['AssimLight','BlackScr', 'CO2air' , 'EC_drain_PC' , 'EnScr' , 'HumDef' , 
                                      'PipeGrow' , 'PipeLow' , 'Rhair' , 'Tair' ,'Tot_PAR' , 'Tot_PAR_Lamps' , 
                                      'VentLee' , 'Ventwind' , 'co2_dos','Water_Quantity','Duration_of_Irrigation',
                                      'Irrigation_Time_Intervals']):
    mae =  tf.keras.metrics.mean_absolute_error(
        greenhouse_climate_valid[name][seq_length:], ghc_valid_prediction[:, idx])
    print(name, mae)

###sequence to sequence dataset

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))

In [ ]:
def to_seq2seq_dataset(series, seq_length=24*12*7, ahead=24*12, target_col=1,
                       batch_size=32, shuffle=False, seed=None):
    ds = to_windows(tf.data.Dataset.from_tensor_slices(series), ahead + 1)
    ds = to_windows(ds, seq_length).map(lambda S: (S[:, 0], S[:, 1:, 1]))
    if shuffle:
        ds = ds.shuffle(1 * batch_size, seed=seed)
    return ds.batch(batch_size)

In [ ]:
seq2seq_train = to_seq2seq_dataset(ghsp_train, shuffle=True, seed=42)
seq2seq_valid = to_seq2seq_dataset(ghsp_valid)

In [ ]:
tf.random.set_seed(42)  
seq2seq_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=[None, greenhouse_sp.shape[1]]),
    tf.keras.layers.LSTM(32, return_sequences=True),
    tf.keras.layers.Dense(12*24)
    # equivalent: tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(14))
    # also equivalent: tf.keras.layers.Conv1D(14, kernel_size=1)
])

In [ ]:
fit_and_evaluate(seq2seq_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.05,ckpt="seq2seq_model")

In [ ]:
tf.random.set_seed(42)  
gru_model = tf.keras.Sequential([
    tf.keras.layers.GRU(256, return_sequences=True, input_shape=[None, greenhouse_sp.shape[1]]),
    tf.keras.layers.Dense(12*24)
])

In [ ]:
fit_and_evaluate(gru_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.01,ckpt="gru_model", epochs=50)

In [ ]:
tf.random.set_seed(42)  
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=[None, greenhouse_sp.shape[1]]),
    tf.keras.layers.LSTM(32, return_sequences=True),

    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(12*24)
])

In [ ]:
fit_and_evaluate(lstm_model, seq2seq_train, seq2seq_valid,
                 learning_rate=0.05,ckpt="lstm_model", epochs=50)

In [ ]:
tf.random.set_seed(42)  
conv_rnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32, kernel_size=4, strides=2,
                           activation="leaky_relu", input_shape=[None, greenhouse_sp.shape[1]]),
    tf.keras.layers.GRU(256, return_sequences=True),
    tf.keras.layers.Dense(12*24)
])

longer_train = to_seq2seq_dataset(ghsp_train, seq_length=12*24*7,
                                       shuffle=True, seed=42)
longer_valid = to_seq2seq_dataset(ghsp_valid, seq_length=12*24*7)
downsampled_train = longer_train.map(lambda X, Y: (X, Y[:, 3::2]))
downsampled_valid = longer_valid.map(lambda X, Y: (X, Y[:, 3::2]))

In [ ]:
fit_and_evaluate(conv_rnn_model, downsampled_train, downsampled_valid,
                 learning_rate=0.01,ckpt="conv_rnn_model", epochs=50)

In [ ]:
tf.random.set_seed(42)  
wavenet_model = tf.keras.Sequential()
wavenet_model.add(tf.keras.layers.InputLayer(input_shape=[None, greenhouse_sp.shape[1]]))
for rate in (1, 2, 4, 8) * 2:
    wavenet_model.add(tf.keras.layers.Conv1D(
        filters=32, kernel_size=2, padding="causal", activation="leaky_relu",
        dilation_rate=rate))
wavenet_model.add(tf.keras.layers.Conv1D(filters=12*24, kernel_size=1))

In [ ]:
fit_and_evaluate(wavenet_model, longer_train, longer_valid,
                 learning_rate=0.01,ckpt="wavenet_model", epochs=30)

In [ ]:
class GatedActivationUnit(tf.keras.layers.Layer):
    def __init__(self, activation="tanh", **kwargs):
        super().__init__(**kwargs)
        self.activation = tf.keras.activations.get(activation)

    def call(self, inputs):
        n_filters = inputs.shape[-1] // 2
        linear_output = self.activation(inputs[..., :n_filters])
        gate = tf.keras.activations.sigmoid(inputs[..., n_filters:])
        return self.activation(linear_output) * gate

In [ ]:
def wavenet_residual_block(inputs, n_filters, dilation_rate):
    z = tf.keras.layers.Conv1D(2 * n_filters, kernel_size=2, padding="causal",
                            dilation_rate=dilation_rate)(inputs)
    z = GatedActivationUnit()(z)
    z = tf.keras.layers.Conv1D(n_filters, kernel_size=1)(z)
    return tf.keras.layers.Add()([z, inputs]), z

In [ ]:
tf.random.set_seed(42)

n_layers_per_block = 3  # 10 in the paper
n_blocks = 1  # 3 in the paper
n_filters = 32  # 128 in the paper
n_outputs = 12*24 *greenhouse_sp.shape[1] # 256 in the paper

inputs = tf.keras.layers.Input(shape=[None, greenhouse_sp.shape[1]])
z = tf.keras.layers.Conv1D(n_filters, kernel_size=2, padding="causal")(inputs)
skip_to_last = []
for dilation_rate in [2**i for i in range(n_layers_per_block)] * n_blocks:
    z, skip = wavenet_residual_block(z, n_filters, dilation_rate)
    skip_to_last.append(skip)

z = tf.keras.activations.relu(tf.keras.layers.Add()(skip_to_last))
z = tf.keras.layers.Conv1D(n_filters, kernel_size=1, activation="relu")(z)
Y_preds = tf.keras.layers.Conv1D(n_outputs, kernel_size=1)(z)

full_wavenet_model = tf.keras.Model(inputs=[inputs], outputs=[Y_preds])

In [ ]:
fit_and_evaluate(full_wavenet_model, longer_train, longer_valid,
                 learning_rate=0.01,ckpt="full_wavenet_model" , epochs=30)

# Statefull LSTM

In [ ]:
length=12*24
def to_dataset_for_stateful_rnn(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(length + 1)).batch(1)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

stateful_train_set = to_dataset_for_stateful_rnn(ghc_train, length)
stateful_valid_set = to_dataset_for_stateful_rnn(ghc_valid,length)
stateful_test_set = to_dataset_for_stateful_rnn(ghc_test, length)

In [ ]:
stateful_train_set.shape
stateful_valid_set.shape
stateful_test_set.shape

In [ ]:
# one way to prepare a batched dataset for a stateful RNN

import numpy as np

def to_non_overlapping_windows(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    return ds.flat_map(lambda window: window.batch(length + 1))

def to_batched_dataset_for_stateful_rnn(sequence, length, batch_size=32):
    parts = np.array_split(sequence, batch_size)
    datasets = tuple(to_non_overlapping_windows(part, length) for part in parts)
    ds = tf.data.Dataset.zip(datasets).map(lambda *windows: tf.stack(windows))
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

bached_stateful_train_set=list(to_batched_dataset_for_stateful_rnn(ghc_train, length=12*24, batch_size=32))
bached_stateful_valid_set=list(to_batched_dataset_for_stateful_rnn(ghc_valid, length=12*24, batch_size=32))
bached_stateful_test_set=list(to_batched_dataset_for_stateful_rnn(ghc_test, length=12*24, batch_size=32))

In [ ]:
print(bached_stateful_train_set)

In [ ]:
bached_stateful_train_set.shape

In [ ]:
tf.random.set_seed(42)  
model = tf.keras.Sequential([
    #tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16,
    #),
    tf.keras.layers.GRU(128, batch_input_shape=[32, 288,44], return_sequences=True, stateful=True),
    tf.keras.layers.Dense(24*12)
])

In [ ]:
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

In [ ]:
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "stateful_model",
    monitor="val_mae",
    save_best_only=True)

In [ ]:
learning_rate=0.05
epochs=10

early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=2, restore_best_weights=True)
opt = Adam(learning_rate=learning_rate, decay=learning_rate / epochs)
model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, #"nadam",
              metrics=["mae"])
history = model.fit(bached_stateful_train_set, validation_data=bached_stateful_valid_set,
                    epochs=epochs, callbacks=[early_stopping_cb,ResetStatesCallback(), model_ckpt])

valid_loss, valid_mae = model.evaluate(stateful_valid_set)
